In [1]:
# now test on FOLIO data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lark

In [3]:
# try a parser for a FOL EBNF grammar to manipulate FOL statements as parse trees and make transforming to CGIF easier
from lark import Lark

fol_grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline | keyword* quantifier* symbol* leftparen* (quantifier symbol)* proposition rightparen* newline | keyword* (quantifier symbol)* leftparen* (quantifier symbol)* proposition rightparen* newline
    proposition: atomicproposition | complexproposition
    complexproposition: keyword* proposition keyword leftparen* (quantifier symbol)* proposition rightparen*
    atomicproposition: leftparen* term* leftparen* term* rightparen*
    !term: (LETTER+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠")* | (DIGIT+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠")*
    !leftparen: "("
    !rightparen: ")"
    !keyword: "∧" | "¬" | "→" | "∨" | "⊕" | "↔" | "⟷"
    !quantifier: "∃" | "∀"
    symbol: LETTER
    newline: /\n/

    %import common.LETTER
    %import common.DIGIT
    %import common.INT -> NUMBER
    %import common.ESCAPED_STRING -> STRING
    %import common.WS
    %ignore WS
"""

parser = Lark(fol_grammar)


def run_turtle(program):
    parse_tree = parser.parse(program)
    for inst in parse_tree.children:
        print("INST:",inst)
        print("************")

def main():
    while True:
        code = input('> ')
        try:
            run_turtle(code)
        except Exception as e:
            print(e)

def test():
    text = """
∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))\n
    """
    run_turtle(text)

if __name__ == '__main__':
    test()
    #main()

INST: Tree(Token('RULE', 'program'), [Tree(Token('RULE', 'stat'), [Tree(Token('RULE', 'quantifier'), [Token('__ANON_8', '∃')]), Tree(Token('RULE', 'symbol'), [Token('LETTER', 'x')]), Tree(Token('RULE', 'leftparen'), [Token('LPAR', '(')]), Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'complexproposition'), [Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'complexproposition'), [Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'atomicproposition'), [Tree(Token('RULE', 'term'), [Token('LETTER', 'C'), Token('LETTER', 'a'), Token('LETTER', 'n'), Token('LETTER', 'i'), Token('LETTER', 'n'), Token('LETTER', 'e')]), Tree(Token('RULE', 'leftparen'), [Token('LPAR', '(')]), Tree(Token('RULE', 'term'), [Token('LETTER', 'x')]), Tree(Token('RULE', 'rightparen'), [Token('RPAR', ')')])])]), Tree(Token('RULE', 'keyword'), [Token('__ANON_1', '∧')]), Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'atomicproposition'), [])])])]), Tree(Token('RULE', 'keyword'), [Token('__ANON_

In [4]:
import sys
import lark
import json

# test how to manipulate Lark parse tree output
def tree_to_json_str(item):
    output = []
    tree_to_json(item, output.append)  # will build output in memory
    return ''.join(output)

def tree_to_json(item, write=None):
    """ Writes a Lark tree as a JSON dictionary. """
    if write is None: write = sys.stdout.write
    _tree_to_json(item, write, 0)

def _tree_to_json(item, write, level):
    indent = '  ' * level
    level += 1
    if isinstance(item, lark.Tree):
        write(f'{indent}{{ "type": "{item.data}", "children": [\n')
        sep = ''
        for child in item.children:
            write(indent)
            write(sep)
            _tree_to_json(child, write, level)
            sep = ',\n'
        write(f'{indent}] }}\n')
    elif isinstance(item, lark.Token):
        # reminder: Lark Tokens are directly strings
        # token attrs include: line, end_line, column, end_column, pos_in_stream, end_pos
        write(f'{indent}{{ "type": "{item.type}", "text": "{item}", "line": {item.line}, "col": {item.column} }}\n')
    else:
        assert False, item  # fall-through

testsent = "∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))\n"
tree  = parser.parse(testsent)
print(tree )

# print with tree_to_json
#tree_to_json(tree)  # will print to stdout

# build JSON string with tree_to_json_str
json_str = tree_to_json_str(tree )
#print(type(json_str))
#print(json_str)

# now convert to json object
parsed_json = json.loads(json_str, strict=False)
#print(type(parsed_json))
#print(parsed_json)

#try to manipulate json object
for key, val in parsed_json.items():
  print("*** KEY:",key, "*** VAL:",val)


print(parsed_json['children'][0]['children'][0]['children'][0]['children'])

Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'program'), [Tree(Token('RULE', 'stat'), [Tree(Token('RULE', 'quantifier'), [Token('__ANON_8', '∃')]), Tree(Token('RULE', 'symbol'), [Token('LETTER', 'x')]), Tree(Token('RULE', 'leftparen'), [Token('LPAR', '(')]), Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'complexproposition'), [Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'complexproposition'), [Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'atomicproposition'), [Tree(Token('RULE', 'term'), [Token('LETTER', 'C'), Token('LETTER', 'a'), Token('LETTER', 'n'), Token('LETTER', 'i'), Token('LETTER', 'n'), Token('LETTER', 'e')]), Tree(Token('RULE', 'leftparen'), [Token('LPAR', '(')]), Tree(Token('RULE', 'term'), [Token('LETTER', 'x')]), Tree(Token('RULE', 'rightparen'), [Token('RPAR', ')')])])]), Tree(Token('RULE', 'keyword'), [Token('__ANON_1', '∧')]), Tree(Token('RULE', 'proposition'), [Tree(Token('RULE', 'atomicproposition'), [])])])]), Tree(Token('RULE', 'ke

In [5]:
# construct TFL from FOL recursively

"""
TFL golden rules:
PLUS: 'yes', 'some', 'is', 'both', 'and', 'then'
MINUS: 'not','every', 'if, 'isn't', 'andn't', 'thenn't'
"""

def iterate_nested_json_for_loop(json_obj, tfl_list):
    for key, value in json_obj.items():
        if isinstance(value, dict):
            iterate_nested_json_for_loop(value, tfl_list)
        elif isinstance(value, list):
            for item in value:
              if isinstance(item, dict):
                if item['type'] == 'quantifier':
                  if item['children'][0]['text'] == "∀":
                    tfl_list.append('-')
                  else:
                    tfl_list.append('+')
                elif item['type'] == 'keyword':
                  # choose PLUS or MINUS sign based on keyword
                  if item['children'][0]['text'] == "∧":
                    tfl_list.append('+')
                  else:
                    tfl_list.append('-')
                elif item['type'] == 'atomicproposition':
                  if item['children']:
                    for elem in item['children']:
                      if elem['type'] == 'term': # verify that we are in the case where there is a literal term in the atomicproposition
                        all_terms = [val['text'] for val in elem['children']]
                        if ',' not in all_terms and len(all_terms) > 1:  # make sure that we are targeting predicates and not terms like (x, y) or (x) or (mike) --> should be flattenShirt for example
                          tfl_list.append('+')
                          tfl_list.append(elem['children'][0]['text'])   # represent predicate by using its first letter, e.g., Recommended(x) --> R
                          tfl_list.append(str(elem['children'][0]['line']))   # in TFL predicates are represented with a letter and a number, e.g., Recommended(x) --> R0
                else:
                  iterate_nested_json_for_loop(item, tfl_list)
        else:
            pass

        #return ''.join(tfl_list)

mylist = []
answer = iterate_nested_json_for_loop(parsed_json, mylist)
print(mylist)

['+', '+', 'C', '1', '+', '-', '+', 'A', '1']


In [6]:
# try a parser for a TFL BNF grammar
tfl_grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline
    proposition: atomicproposition | complexproposition
    complexproposition: leftparen proposition rightparen | proposition plus proposition | proposition minus proposition
    atomicproposition: term
    leftparen: "("
    rightparen: ")"
    plus: "+"
    minus: "-"
    term: plus T n | minus T n
    T: LETTER
    n: NUMBER
    newline: /\n/

    %import common.LETTER
    %import common.INT -> NUMBER
    %import common.WS
    %ignore WS
"""

#parser = Lark(tfl_grammar)


def run_turtle(program):
    parse_tree = parser.parse(program)
    for inst in parse_tree.children:
        print("INST:",inst)
        print("************")

def main():
    while True:
        code = input('> ')
        try:
            run_turtle(code)
        except Exception as e:
            print(e)

def test():
    text = """
       +s0 + +M0 - (-L0 + -m0)\n
       (+s0 + +M0) - (-L0 + -m0)\n
       +s0 - +M0\n
       +s0 + -M0\n
       +s0 - -M0\n
       -s0 + +M0\n
       -s0 + -M0\n
       -H0 - -G0\n
       -s2 - +M1\n
       (-s2 - +M1)\n
       -s2 - +M1 - +M1\n
       (-s2 - +M1) - (+M1)\n
    """
    run_turtle(text)

if __name__ == '__main__':
    test()
    #main()

UnexpectedCharacters: No terminal matches '+' in the current parser context, at line 2 col 8

       +s0 + +M0 - (-L0 + -m0)
       ^
Expected one of: 
	* __ANON_3
	* __ANON_2
	* __ANON_4
	* __ANON_9
	* __ANON_10
	* RPAR
	* __ANON_7
	* DIGIT
	* __ANON_5
	* __ANON_6
	* LETTER
	* LPAR
	* __ANON_1
	* __ANON_8


In [7]:
# construct TFL+ from FOL recursively

"""
TFL+ golden rules:
PLUS: 'yes', 'some', 'is', 'both', 'and', 'then'
MINUS: 'not','every', 'if, 'isn't', 'andn't', 'thenn't'
PARENTHESIS
SUPERSCRIPTS: 0 = FORALL, 1 = MOST, 2 = SOME
"""

def construct_tfl_plus_from_fol_json(json_obj, tfl_list, subscript):
    for key, value in json_obj.items():
        if isinstance(value, dict):
            construct_tfl_plus_from_fol_json(value, tfl_list, subscript)
        elif isinstance(value, list):
            for item in value:
              if isinstance(item, dict):
                if item['type'] == 'quantifier':
                  if item['children'][0]['text'] == "∀":
                    tfl_list.append('-')
                    subscript = 0
                  else:
                    tfl_list.append('+')
                    subscript = 1
                elif item['type'] == 'keyword':
                  # choose PLUS or MINUS sign based on keyword
                  if item['children'][0]['text'] == "∧":
                    tfl_list.append('+')
                  else:
                    tfl_list.append('-')
                elif item['type'] == 'atomicproposition':
                  if item['children']:
                    for elem in item['children']:
                      if elem['type'] == 'term': # verify that we are in the case where there is a literal term in the atomicproposition
                        all_terms = [val['text'] for val in elem['children']]
                        if ',' not in all_terms and len(all_terms) > 1:  # make sure that we are targeting predicates and not terms like (x, y) or (x) or (mike) --> should be flattenShirt for example
                          tfl_list.append('+')
                          tfl_list.append(elem['children'][0]['text'])   # represent predicate by using its first letter, e.g., Recommended(x) --> R
                          tfl_list.append(str(subscript))   # in TFL predicates are represented with a letter and a number, e.g., Recommended(x) --> R0
                      if elem['type'] == 'leftparen':  # keep parenthesis
                        tfl_list.append(elem['children'][0]['text'])
                      if elem['type'] == 'rightparen':
                        tfl_list.append(elem['children'][0]['text'])
                elif item['type'] == 'leftparen':
                  tfl_list.append(item['children'][0]['text'])
                elif item['type'] == 'rightparen':
                  tfl_list.append(item['children'][0]['text'])
                else:
                  construct_tfl_plus_from_fol_json(item, tfl_list, subscript)
        else:
            pass

        #return ''.join(tfl_list)

mylist = []
subscript = 2
answer = construct_tfl_plus_from_fol_json(parsed_json, mylist, subscript)
print(mylist)

['+', '(', '+', 'C', '1', '(', ')', '+', '-', '+', 'A', '1', '(', ')', ')']


In [8]:
# test automatic TFL  transformation from FOL

def fol_to_tfl(fol_string):
  print("*** FOL STRING:", fol_string)
  parsed_fol_example = parser.parse(fol_string)
  # build JSON string with tree_to_json_str
  json_str_example = tree_to_json_str(parsed_fol_example)
  # now convert to json object
  parsed_json_example = json.loads(json_str_example, strict=False)
  tfl_example = []
  iterate_nested_json_for_loop(parsed_json_example, tfl_example)
  tfl_example_str = ''.join(tfl_example)
  return tfl_example_str


fol_string = "∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))\n"
custom_tfl_string = fol_to_tfl(fol_string)

print("*** FOL EXAMPLE ***")
print(fol_string)
print("*** TFL RESULT ***")
print(custom_tfl_string)

*** FOL STRING: ∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))

*** FOL EXAMPLE ***
∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))

*** TFL RESULT ***
++C1+-+A1


In [9]:
# clgc transformations

def fol_to_tfl_plus(fol_string):
  print("*** FOL STRING:", fol_string)
  parsed_fol_example = parser.parse(fol_string)
  # build JSON string with tree_to_json_str
  json_str_example = tree_to_json_str(parsed_fol_example)
  # now convert to json object
  parsed_json_example = json.loads(json_str_example, strict=False)
  tfl_example = []
  subscript=2
  construct_tfl_plus_from_fol_json(parsed_json_example, tfl_example, subscript)
  tfl_example_str = ''.join(tfl_example)
  tfl_example_str = tfl_example_str.replace("()","")
  return tfl_example_str


def fol_to_clif(fol_string):
  clif_string = ""
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      clif_string += "forall "
    elif fol_string[i] == "⊕":
      clif_string += "xor"
    elif fol_string[i] == "→":
      clif_string += "implies"
    elif fol_string[i] == "¬":
      clif_string += "not "
    elif fol_string[i] == "∃":
      clif_string += "exists "
    elif fol_string[i] == "∧":
      clif_string += "and"
    elif fol_string[i] == "∨":
      clif_string += "or"
    else:
      clif_string += fol_string[i]
  return clif_string.lower()


def fol_to_cgif(fol_string):
  cgif_string = ""
  cgif_string += "[" # always start a CGIF statement with [
  symbols = [] # list to store quantified symbols to keep track and add proper syntax to them
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      cgif_string += "@every *"
      symbols.append(fol_string[i+1])
    elif fol_string[i] == "∧" or fol_string[i] == "→" or fol_string[i] == "∨" or fol_string[i] == "⊕" or fol_string[i] == "↔" or fol_string[i] == "⟷":
      cgif_string += ""
    elif fol_string[i] == "¬":
      cgif_string += "~"
    elif fol_string[i] == "∃":
      cgif_string += "*"
      symbols.append(fol_string[i+1])
    elif fol_string[i] == "(":
      cgif_string += "[("
    elif fol_string[i] == ")":
      cgif_string += ")]"
    elif fol_string[i] == ",":
      cgif_string += " "
    else:
      if (fol_string[i] in symbols and (fol_string[i-1] == "(" or fol_string[i-1] == "∀" or fol_string[i-1] == "∃" or fol_string[i-1] == ",")):
        cgif_string += "?"
        cgif_string += fol_string[i]
      else:
        cgif_string += fol_string[i]
  cgif_string += "]" # always end a CGIF statement with ]
  cgif_string = cgif_string.replace("*?", "*")
  cgif_string = cgif_string.replace("[~", "~[")
  return cgif_string.lower()


def fol_to_clingo(fol_string):
  custom_cgif_string = ""
  symbols = [] # list to store quantified symbols to keep track and add proper syntax to them
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      custom_cgif_string += "forall"
      symbols.append(fol_string[i+1])
    elif fol_string[i] == "⊕":
      custom_cgif_string += "^"
    elif fol_string[i] == "→":
      custom_cgif_string += "-:"
    elif fol_string[i] == "¬":
      custom_cgif_string += "not"
    elif fol_string[i] == "∃":
      custom_cgif_string += ""
      symbols.append(fol_string[i+1])
    elif fol_string[i] == "∧":
      custom_cgif_string += ","
    elif fol_string[i] == "∨":
      custom_cgif_string += "|"
    elif (fol_string[i] in symbols and (fol_string[i-1] == "∀" or fol_string[i-1] == "∃" or fol_string[i-1] == ",")):
      custom_cgif_string += ""  # eliminate quantifier symbols (e.g., x in ∀x)
    else:
      custom_cgif_string += fol_string[i]
  return custom_cgif_string.lower()


"""
minifol2 is the same as minifol with the removal of "some" entirely for the "∃" symbol (does not replace it with anything else)
"""

def fol_to_minifol2(fol_string):
  custom_cgif_string = ""
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      custom_cgif_string += "all:"
    elif fol_string[i] == "⊕":
      custom_cgif_string += "^"
    elif fol_string[i] == "→":
      custom_cgif_string += ":-"
    elif fol_string[i] == "¬":
      custom_cgif_string += "~"
    elif fol_string[i] == "∃":
      custom_cgif_string += ""
    elif fol_string[i] == "∧":
      custom_cgif_string += "&"
    elif fol_string[i] == "∨":
      custom_cgif_string += "|"
    else:
      custom_cgif_string += fol_string[i]
  return custom_cgif_string.lower()


"""
minifol3 is the same as minifol with the replacement of the "¬" symbol by "not"
"""

def fol_to_minifol3(fol_string):
  custom_cgif_string = ""
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      custom_cgif_string += "all:"
    elif fol_string[i] == "⊕":
      custom_cgif_string += "^"
    elif fol_string[i] == "→":
      custom_cgif_string += ":-"
    elif fol_string[i] == "¬":
      custom_cgif_string += "not"
    elif fol_string[i] == "∃":
      custom_cgif_string += "some:"
    elif fol_string[i] == "∧":
      custom_cgif_string += "&"
    elif fol_string[i] == "∨":
      custom_cgif_string += "|"
    else:
      custom_cgif_string += fol_string[i]
  return custom_cgif_string.lower()

"""
minifol4 is the same as minifol with the replacement of the "∧" symbol by ","
"""

def fol_to_minifol4(fol_string):
  custom_cgif_string = ""
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      custom_cgif_string += "all:"
    elif fol_string[i] == "⊕":
      custom_cgif_string += "^"
    elif fol_string[i] == "→":
      custom_cgif_string += ":-"
    elif fol_string[i] == "¬":
      custom_cgif_string += "~"
    elif fol_string[i] == "∃":
      custom_cgif_string += "some:"
    elif fol_string[i] == "∧":
      custom_cgif_string += ","
    elif fol_string[i] == "∨":
      custom_cgif_string += "|"
    else:
      custom_cgif_string += fol_string[i]
  return custom_cgif_string.lower()


def fol_to_minifol(fol_string):
  custom_cgif_string = ""
  for i in range(len(fol_string)):
    if fol_string[i] == "∀":
      custom_cgif_string += "all:"
    elif fol_string[i] == "⊕":
      custom_cgif_string += "^"
    elif fol_string[i] == "→":
      custom_cgif_string += ":-"
    elif fol_string[i] == "¬":
      custom_cgif_string += "~"
    elif fol_string[i] == "∃":
      custom_cgif_string += "some:"
    elif fol_string[i] == "∧":
      custom_cgif_string += "&"
    elif fol_string[i] == "∨":
      custom_cgif_string += "|"
    else:
      custom_cgif_string += fol_string[i]
  return custom_cgif_string.lower()

fol_string = "∀x (BelieveIn(x, santaClaus) ⊕ ThinkMadeUp(x, santaClaus))"
target_custom_cgif_string = "([all{var:x}]) & ([believein{var:x var:santa claus}] ^ [thinkmadeup{var:x var:santa claus}])"
custom_cgif_string = fol_to_minifol(fol_string)

print("RESULT:", custom_cgif_string)



RESULT: all:x (believein(x, santaclaus) ^ thinkmadeup(x, santaclaus))


In [38]:
# load pilot train, and test data
# extract syllogisms from pilot json
import json

with open('/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/pilot_fol.json') as json_data:
    pilot_data = json.load(json_data)
    json_data.close()

with open('/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/train_fol.json') as json_data:
    train_data = json.load(json_data)
    json_data.close()

with open('/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test_fol.json') as json_data:
    test_data = json.load(json_data)
    json_data.close()

# clean FOL syllogisms from ";" and "∴" characters
def clean_fol_syllogisms(fol_string):
  fol_string = fol_string.replace(" ;", '\n').replace(" ∴", '\n').replace("_", '')
  return fol_string

print("*** PILOT DATA ***")
print(pilot_data[0])

for item in pilot_data:
 item['fol'] = clean_fol_syllogisms(item['fol'])

print(pilot_data[0])
print(pilot_data[1])

print("*** TRAIN DATA ***")
print(train_data[0])

for item in train_data:
 item['fol'] = clean_fol_syllogisms(item['fol'])

print(train_data[0])
print(train_data[1])

print("*** TEST DATA ***")
print(test_data[0])

for item in test_data:
 premises =  " ;".join([p for p in item['premises']])  # transform list of premises into string of premises separated by ";"
 item['fol'] = premises + ' ∴' + item['conclusion'] + '\n' if item['conclusion'] else premises + '\n'
 item['fol'] = clean_fol_syllogisms(item['fol'])

print(test_data[0])
print(test_data[1])

*** PILOT DATA ***
{'id': '0', 'syllogism': 'Not all canines are aquatic creatures known as fish. It is certain that no fish belong to the class of mammals. Therefore, every canine falls under the category of mammals.', 'validity': False, 'plausibility': True, 'fol': '∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x)) ∴ ∀x (Fish(x) → ¬MammalThereforeEveryCanineFallUnderTheCategoryOfMammal(x))\n'}
{'id': '0', 'syllogism': 'Not all canines are aquatic creatures known as fish. It is certain that no fish belong to the class of mammals. Therefore, every canine falls under the category of mammals.', 'validity': False, 'plausibility': True, 'fol': '∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))\n ∀x (Fish(x) → ¬MammalThereforeEveryCanineFallUnderTheCategoryOfMammal(x))\n'}
{'id': '1', 'syllogism': 'All birds lay eggs. All chickens lay eggs. All chickens are birds.', 'validity': False, 'plausibility': True, 'fol': '∀x (Bird(x) → Egg(x))\n ∀x (Chicken(x) → Egg(x))\n ∀x (Chicken(x) → Bird(x))\n'}


In [ ]:
# execute clgc pipeline and transform fol to other grammars
for item in pilot_data:
  item['clif'] = fol_to_clif(item['fol'])
  item['cgif'] = fol_to_cgif(item['fol'])
  item['clingo'] = fol_to_clingo(item['fol'])
  item['tflplus'] = fol_to_tfl_plus(item['fol'])
  item['minifol2'] = fol_to_minifol2(item['fol'])

print("*** PILOT DATA AFTER CLGC TRANSFORMATIONS ***")
print(pilot_data[0])
print(pilot_data[1])

*** FOL STRING: ∃x (Canine(x) ∧ ¬AquaticCreatureKnownAsFish(x))
 ∀x (Fish(x) → ¬MammalThereforeEveryCanineFallUnderTheCategoryOfMammal(x))

*** FOL STRING: ∀x (Bird(x) → Egg(x))
 ∀x (Chicken(x) → Egg(x))
 ∀x (Chicken(x) → Bird(x))

*** FOL STRING: ∃x (BridgeThat(x) ∧ ConsideredArtifact(x))
 ∀x (ItemCategorizedAsStructure(x) → Artifact(x))
 ∀x (Bridge(x) → Structure(x))

*** FOL STRING: ∀x (ApparatuThat(x) → TimeCanBeClassifiedAsAnInstrument(x))
 ∀x (ToolSpecificallyDesignedForMeasurement(x) → ByDefinitionAnInstrument(x))
 ∀x (EveryApparatuThatIndicateTime(x) → ToolSpecificallyDesignedForMeasurement(x))

*** FOL STRING: ∀x (There(x) → ActivitieThatCanBeClassifiedAsCovering(x))
 ∀x (ActOfProtecting(x) → Covering(x))
 ∀x (ActOfProtecting(x) → Activity(x))

*** FOL STRING: ∃x (Sadnesse(x) ∧ NotTree(x))
 ∀x (Sadnesse(x) → Emotion(x))
 ∀x (Tree(x) → ¬Emotion(x))

*** FOL STRING: ∀x (CreatureClassifiedAsAnArthropod(x) → Invertebrate(x))
 ∃x (Invertebrate(x) ∧ KnownToBeCentipede(x))
 ∀x (Every

In [ ]:
for item in train_data:
  item['clif'] = fol_to_clif(item['fol'])
  item['cgif'] = fol_to_cgif(item['fol'])
  item['clingo'] = fol_to_clingo(item['fol'])
  item['tflplus'] = fol_to_tfl_plus(item['fol'])
  item['minifol2'] = fol_to_minifol2(item['fol'])

print("*** TRAIN DATA AFTER CLGC TRANSFORMATIONS ***")
print(train_data[0])
print(train_data[1])

*** FOL STRING: ∀x (Car(x) → TypeOfVehicle(x))
 ∀x (Animal(x) → ¬Car(x))
 ∀x (NoAnimalCanBeAVehicle(x))

*** FOL STRING: ∀x (NothingThat(x) → SodaAJuice(x))
 ∃x (ThingThat(x) ∧ BeverageJuice(x))
 ∀x (OnlyLogicalConclusion(x) → ThatSomeBeverageNotSoda(x))

*** FOL STRING: ∀x (EverythingThat(x) → PlanetACelestialBody(x))
 ∀x (AnythingThat(x) → SunACelestialBody(x))
 ∀x (ThereExistAtLeastOneSunThat(x) → Planet(x))

*** FOL STRING: ∀x (Cat(x) → InvisibleCreature(x))
 ∃x (Cat(x) ∧ Animal(x))
 ∀x (APortionOfAnimal(x) → Invisible(x))

*** FOL STRING: ∃x (NoCapitalCitieWhich(x) ∧ Ocean(x))
 ∀x (SelectFewLargeCitie(x) → ClassifiedAsCapitalCitie(x))
 ∀x (It(x) → ClearThatSomeLargeCitieNotOcean(x))

*** FOL STRING: ∀x (City(x) → Location(x))
 ∀x (AnythingThat(x) → LocationACapitalCity(x))
 ∀x (EveryCapitalCity(x) → City(x))

*** FOL STRING: ∀x (Star(x) → ¬Meteor(x))
 ∀x (Meteor(x) → WithoutExceptionPlanet(x))
 ∃x (Planet(x) ∧ Star(x))

*** FOL STRING: ∀x (SingleCat(x) → Animal(x))
 ∀x (AnythingTh

In [39]:
for item in test_data:
  item['clif'] = fol_to_clif(item['fol'])
  item['cgif'] = fol_to_cgif(item['fol'])
  item['clingo'] = fol_to_clingo(item['fol'])
  item['tflplus'] = fol_to_tfl_plus(item['fol'])
  item['minifol2'] = fol_to_minifol2(item['fol'])

print("*** TEST DATA AFTER CLGC TRANSFORMATIONS ***")
print(test_data[0])
print(test_data[1])

*** FOL STRING: ∀x (Bikes(x) → ¬Calledcars(x))
∀x (Bike(x) → Vehicle(x))
∃x (Vehicles(x) ∧ Bikes(x))

*** FOL STRING: ∃x (Objects(x) ∧ Booksnotdigitalfiles(x))
∀x (Book(x) → Itempages(x))
∃x (Itemspages(x) ∧ Digitalfiles(x))

*** FOL STRING: ∀x (Object(x) → CanFly(x))
∃x (Boats(x) ∧ ¬CanFly(x))
∃x (Boats(x) ∧ Notobjects(x))

*** FOL STRING: ∀x (Fruit(x) → Food(x))
∀x (Nothing(x) → Vegetablefood(x))
∀x (Vegetable(x) → Fruit(x))

*** FOL STRING: ∀x (Individualselephants(x) → ¬Peopleenrolledinkindergarten(x))
∀x (Fiveyearolds(x) → Peopleenrolledinkindergarten(x))
∃x (Fiveyearolds(x) ∧ Notelephants(x))

*** FOL STRING: ∀x (Fruits(x) → Roundobjectscitruspeel(x))
∀x (Atleastoneorange(x) → ¬Roundobjectcitruspeel(x))
∀x (Fruit(x) → Orange(x))

*** FOL STRING: ∀x (Noanimal(x) → Rock(x))
∃x (Rocks(x) ∧ Plants(x))
∃x (Animals(x) ∧ Plants(x))

*** FOL STRING: ∀x (Piecefurniture(x) → ¬Calledplanet(x))
∀x (Chair(x) → Piecefurniture(x))
∀x (Nochair(x) → Planet(x))

*** FOL STRING: ∀x (Vehicle(x) → Co

In [ ]:
# export pilot and train datasets
import json


with open("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/pilot_clgc.json", 'w') as final:
  json.dump(pilot_data, final)

with open("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/train_clgc.json", 'w') as final:
  json.dump(train_data, final)

In [40]:
# export test dataset
import json


with open("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test_clgc.json", 'w') as final:
  json.dump(test_data, final)


In [41]:
# apply SEF to categorize syllogisms in the pilot and train datasets
def categorize_syllogism(statements):
  statements = statements.lower()
  list_of_statements = statements.split('\n')
  list_of_statements = list(filter(None, list_of_statements)) # remove empty strings from list
  categorical_keywords = ["all", "any", "some", "no", "few", "most", "none", "several"]  # define universal and existential keywords for categorical syllogisms
  if "∨" in statements or "⊕" in statements:
    return "disjunctive"
  elif len(list_of_statements) > 3:
    return "complex"
  elif any(x in statements for x in categorical_keywords):
    return "categorical"
  else:
    return "hypothetical"

In [ ]:
import json

# add SEF entry to pilot and train datasets
with open('/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/pilot_clgc.json') as json_data:
    pilot_data = json.load(json_data)
    json_data.close()

with open('/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/train_clgc.json') as json_data:
    train_data = json.load(json_data)
    json_data.close()

In [42]:
import json

# add SEF entry to test dataset
with open('/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test_clgc.json') as json_data:
    test_data = json.load(json_data)
    json_data.close()

In [ ]:
# apply sef to pilot data
for item in pilot_data:
  item['sef'] = categorize_syllogism(item['syllogism'])

# apply sef to train data
for item in train_data:
  item['sef'] = categorize_syllogism(item['syllogism'])

In [43]:
# apply sef to test data
for item in test_data:
  item['sef'] = categorize_syllogism(item['syllogism'])

In [44]:
# generate statistics for each sef category
def count_syllogisms(data_dict):
  syllogisms = {'hypothetical': 0, 'disjunctive': 0, 'categorical': 0, 'complex': 0}
  for item in data_dict:
    syllogisms[item['sef']] += 1
  return syllogisms

In [ ]:
print("*** PILOT DATA SEF STATISTICS ***")
print(count_syllogisms(pilot_data))

print("*** TRAIN DATA SEF STATISTICS ***")
print(count_syllogisms(train_data))

In [45]:
print("*** TEST DATA SEF STATISTICS ***")
print(count_syllogisms(test_data))

*** TEST DATA SEF STATISTICS ***
{'hypothetical': 2, 'disjunctive': 0, 'categorical': 189, 'complex': 0}


In [ ]:
# export pilot and train datasets with sef classification
with open("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/pilot_sef.json", 'w') as final:
  json.dump(pilot_data, final)

with open("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/train_sef.json", 'w') as final:
  json.dump(train_data, final)

In [46]:
# export test dataset with sef classification
with open("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test_sef.json", 'w') as final:
  json.dump(test_data, final)

In [ ]:
import pandas as pd

# apply sef on train, valid and test frozen splits
train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/train.csv")
val = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/val.csv")
test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test.csv")

In [ ]:
train["sef"] = train["syllogism"].apply(categorize_syllogism)
val["sef"] = val["syllogism"].apply(categorize_syllogism)
test["sef"] = test["syllogism"].apply(categorize_syllogism)

# generate sef statistics
print("*** TRAIN SPLIT SEF STATISTICS ***")
print(train['sef'].value_counts())

print("*** VAL SPLIT SEF STATISTICS ***")
print(val['sef'].value_counts())

print("*** TEST SPLIT SEF STATISTICS ***")
print(test['sef'].value_counts())

*** TRAIN SPLIT SEF STATISTICS ***
sef
categorical     547
hypothetical     14
Name: count, dtype: int64
*** VAL SPLIT SEF STATISTICS ***
sef
categorical     368
hypothetical      7
Name: count, dtype: int64
*** TEST SPLIT SEF STATISTICS ***
sef
categorical     101
hypothetical      3
Name: count, dtype: int64


In [ ]:
# export train, val and test splits with sef classification
train.to_csv('train.csv', index=False)
val.to_csv('val.csv', index=False)
test.to_csv('test.csv', index=False)

In [48]:
import pandas as pd

# convert test json to dataframe
subtask1_test = pd.read_json("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test_sef.json")

subtask1_test.head()

,id,syllogism,premises,conclusion,fol,clif,cgif,clingo,tflplus,minifol2,sef
0,bff2af61-d4b0-4147-8a5b-ff4fe1892559,There are no bikes that can be called cars. It...,"[∀x (Bikes(x) → ¬Called_cars(x)), ∀x (Bike(x) ...",∃x (Vehicles(x) ∧ Bikes(x)),∀x (Bikes(x) → ¬Calledcars(x))\n∀x (Bike(x) → ...,forall x (bikes(x) implies not calledcars(x))\...,[@every *x [(bikes[(?x)] ~calledcars[(?x)])]\...,forall (bikes(x) -: notcalledcars(x))\nforall ...,-(+B0--+C0)-(+B0-+V0)+(+V1++B1),all:x (bikes(x) :- ~calledcars(x))\nall:x (bik...,categorical
1,f36a4ca3-3b69-4869-a152-deaa7e0fdad4,There exist some objects that are books which ...,"[∃x (Objects(x) ∧ Books_not_digital_files(x)),...",∃x (Items_pages(x) ∧ Digital_files(x)),∃x (Objects(x) ∧ Booksnotdigitalfiles(x))\n∀x ...,exists x (objects(x) and booksnotdigitalfiles(...,[*x [(objects[(?x)] booksnotdigitalfiles[(?x)...,"(objects(x) , booksnotdigitalfiles(x))\nforal...",+(+O1++B1)-(+B0-+I0)+(+I1++D1),x (objects(x) & booksnotdigitalfiles(x))\nall:...,categorical
2,e773bd8c-fa53-4e9c-8ec6-7d978e0601ac,Every single object can fly. It is known that ...,"[∀x (Object(x) → Can_Fly(x)), ∃x (Boats(x) ∧ ¬...",∃x (Boats(x) ∧ Not_objects(x)),∀x (Object(x) → CanFly(x))\n∃x (Boats(x) ∧ ¬Ca...,forall x (object(x) implies canfly(x))\nexists...,[@every *x [(object[(?x)] canfly[(?x)])]\n*x ...,"forall (object(x) -: canfly(x))\n (boats(x) , ...",-(+O0-+C0)+(+B1+-+C1)+(+B1++N1),all:x (object(x) :- canfly(x))\nx (boats(x) & ...,categorical
3,a2fcf47a-5df0-405d-86ce-ed8629f387a9,Every single fruit is a food. Nothing that is ...,"[∀x (Fruit(x) → Food(x)), ∀x (Nothing(x) → Veg...",∀x (Vegetable(x) → Fruit(x)),∀x (Fruit(x) → Food(x))\n∀x (Nothing(x) → Vege...,forall x (fruit(x) implies food(x))\nforall x ...,[@every *x [(fruit[(?x)] food[(?x)])]\n@every...,forall (fruit(x) -: food(x))\nforall (nothing(...,-(+F0-+F0)-(+N0-+V0)-(+V0-+F0),all:x (fruit(x) :- food(x))\nall:x (nothing(x)...,categorical
4,6e5f055f-06e4-40f3-a45b-52b7d9292d60,There are no individuals who are elephants tha...,[∀x (Individuals_elephants(x) → ¬People_enroll...,∃x (Five_year_olds(x) ∧ Not_elephants(x)),∀x (Individualselephants(x) → ¬Peopleenrolledi...,forall x (individualselephants(x) implies not ...,[@every *x [(individualselephants[(?x)] ~peop...,forall (individualselephants(x) -: notpeopleen...,-(+I0--+P0)-(+F0-+P0)+(+F1++N1),all:x (individualselephants(x) :- ~peopleenrol...,categorical


In [49]:
subtask1_test["sef"] = subtask1_test["syllogism"].apply(categorize_syllogism)

# generate sef statistics
print("*** TEST SPLIT SEF STATISTICS ***")
print(subtask1_test['sef'].value_counts())

*** TEST SPLIT SEF STATISTICS ***
sef
categorical     189
hypothetical      2
Name: count, dtype: int64


In [50]:
# export test dataset with sef classification
subtask1_test.to_csv('subtask1_test.csv', index=False)